# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal de clasificación con **Keras 3** sobre Fashion-MNIST (10 clases, imágenes 28×28). Ahora resolverás un problema diferente con el mismo flujo de trabajo: **Olivetti Faces**.

El dataset Olivetti Faces contiene **400 imágenes de rostros en escala de grises de 64×64 píxeles**, correspondientes a **40 personas diferentes** (10 imágenes por persona).

---


---
## 0. Importar librerías

Usaremos:

- **Keras 3** para construir y entrenar la red.
- **scikit-learn** para cargar Olivetti Faces y dividir los datos.
- **NumPy** para manipular arrays.
- **Matplotlib** para visualizar imágenes.


In [ ]:
import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: Olivetti Faces

Olivetti Faces contiene:

- **400 imágenes**
- imágenes de **64 × 64 píxeles**
- escala de grises
- **40 personas**
- **10 imágenes por persona**

La etiqueta indica la identidad de la persona:

```text
0, 1, 2, ..., 39
```

Cada imagen ya viene representada con valores flotantes aproximadamente en el rango:

$$
[0,1]
$$

Por eso, en este ejercicio **no necesitamos normalizar dividiendo por 255**.


In [ ]:

faces = fetch_olivetti_faces(
    shuffle=True,
    random_state=SEED
)

X = faces.images
y = faces.target

print("X:", X.shape)
print("y:", y.shape)
print("Valor mínimo:", X.min())
print("Valor máximo:", X.max())
print("Número de clases:", len(np.unique(y)))


## 1.1 Dividir entrenamiento y prueba

Como solo tenemos 10 imágenes por persona, debemos asegurarnos de que todas las identidades aparezcan tanto en entrenamiento como en prueba.

Usaremos una división **estratificada**:

```text
80% entrenamiento
20% prueba
```

La opción:

```python
stratify=y
```

mantiene la proporción de cada clase en ambos conjuntos.

Esta parte está resuelta.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nPersonas en train:", len(np.unique(y_train)))
print("Personas en test :", len(np.unique(y_test)))


### Pregunta 1

Cada imagen tiene tamaño:

$$
64 \times 64
$$

Cuando apliquemos `Flatten()`, ¿cuántos valores tendrá cada ejemplo?

**Respuesta:**
Cada ejemplo tendrá $64 \times 64 = 4,096$ valores continuos tras la vectorización.


---
## 1.2 Visualizar algunas imágenes

Esta parte está resuelta.

Observa que varias fotografías corresponden a la misma persona con pequeñas variaciones de expresión, iluminación y orientación.


In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Persona {y_train[i]}")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Usaremos una arquitectura fully-connected similar a la vista en clase:

```text
Imagen 64×64
     ↓
Flatten
     ↓
4096 valores
     ↓
Dense(64)
     ↓
ReLU
     ↓
Dense(40)
     ↓
Logits
```

La última capa debe tener una salida por cada clase.

> **Importante:** no añadas `Softmax` al final. Trabajaremos directamente con **logits**.


In [ ]:
# TODO 1 — Completar la arquitectura.

model = keras.Sequential([
    keras.Input(shape=(64, 64)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(40)
])

model.summary()


In [ ]:
# Validación TODO 1
assert isinstance(model, keras.Model)
expected_params = (4096 * 64 + 64) + (64 * 40 + 40)
assert model.count_params() == expected_params
print("✓ Arquitectura correcta")


### Pregunta 2

¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:**
Porque las capas `Dense` exigen como entrada un vector 1D de características. `Flatten()` transforma la matriz bidimensional $64 \times 64$ en un vector plano de 4,096 elementos continuos sin alterar los datos.

### Pregunta 3

¿Por qué la última capa tiene **40 neuronas**?

**Respuesta:**
Porque el dataset Olivetti Faces contiene imágenes de exactamente 40 personas diferentes ($K = 40$ clases disjuntas).

### Pregunta 4

¿Qué función cumple `ReLU`?

**Respuesta:**
Aporta no linealidad ($f(x)=\max(0, x)$), permitiendo a la red aprender fronteras de decisión complejas en el espacio latente.


---
# 3. Configurar el entrenamiento

Las etiquetas son números enteros entre:

```text
0 y 39
```

Por eso usaremos:

```python
SparseCategoricalCrossentropy
```

Además:

- el modelo produce **logits**,
- usaremos **Adam**,
- mediremos **accuracy**.

Completa `model.compile()`.


In [ ]:
lr_rate = 0.001

# TODO 2 — Configurar loss, optimizer y métrica.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


### Pregunta 5

¿Por qué usamos:

```python
from_logits=True
```

en la función de pérdida?

**Respuesta:**
Porque la capa final `Dense(40)` produce logits directamente sin Softmax. Con `from_logits=True`, la pérdida se calcula usando la formulación Log-Sum-Exp analítica combinada, evitando inestabilidades numéricas de punto flotante.


---
# 4. Entrenar el modelo

Usaremos:

```python
epochs = 30
batch_size = 32
```

El dataset es pequeño, por lo que el entrenamiento debería ser rápido.

Keras realizará internamente:

```text
Forward pass
      ↓
Calcular loss
      ↓
Backpropagation
      ↓
Actualizar parámetros con Adam
```

Completa la llamada a `fit()`.


In [ ]:
epochs = 30
batch_size = 32

# TODO 3 — Entrenar el modelo.
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)


### Pregunta 6

¿Qué representa una **epoch**?

**Respuesta:**
Una época corresponde a una pasada completa por las 320 imágenes del conjunto de entrenamiento.

### Pregunta 7

¿Qué significa `batch_size=32`?

**Respuesta:**
Significa que la red toma lotes de 32 imágenes para estimar el gradiente y actualizar sus pesos en cada iteración (10 pasos por época).


---
## 4.1 Visualizar el entrenamiento

Esta parte está resuelta.

El dataset tiene muy pocos ejemplos comparado con el número de parámetros de la red.

Observa cuidadosamente la diferencia entre entrenamiento y validación.


In [ ]:
# Visualización del entrenamiento
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Pérdida por época")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.title("Exactitud por época")
plt.xlabel("Época")
plt.ylabel("Exactitud")
plt.legend()

plt.tight_layout()
plt.show()


### Pregunta 8

Observando la gráfica:

- ¿El accuracy de entrenamiento sigue aumentando?
- ¿El accuracy de validación se comporta igual?
- ¿Observas evidencia de **overfitting**?

**Respuesta:**
- El accuracy de entrenamiento asciende rápidamente hasta valores cercanos al $95-100\%$.
- El accuracy de validación se estanca alrededor de $\approx 60-70\%$.
- **Evidencia clara de overfitting:** la red memoriza las pocas muestras de entrenamiento por individuo pero exhibe dificultad para generalizar ante nuevas variaciones de pose o iluminación en test.


---
# 5. Evaluar el modelo

Ahora evaluaremos el modelo sobre imágenes que no fueron utilizadas para actualizar sus parámetros.

Completa `evaluate()`.


In [ ]:
# TODO 4 — Evaluar.
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Tenemos **40 clases**, pero solamente **320 imágenes de entrenamiento**.

¿Crees que la cantidad de datos es grande o pequeña para una red con cientos de miles de parámetros?

**Respuesta:**
La cantidad de datos es **sumamente pequeña**. Disponer de apenas 8 ejemplos por clase (320 en total) para ajustar más de 264,000 parámetros provoca que la capacidad representacional del modelo exceda la evidencia empírica disponible, conduciendo a sobreajuste severo sin técnicas de regularización o data augmentation.


---
# 6. Logits y Softmax

La salida de nuestro modelo tiene forma:

```text
(batch_size, 40)
```

Cada uno de los 40 valores corresponde al **logit** asociado a una persona.

Los logits no son probabilidades.

Para convertirlos en probabilidades usamos:

```python
keras.ops.softmax(...)
```


In [ ]:
index = 0

image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Persona real: {true_class}")
plt.axis("off")
plt.show()

# Añadimos dimensión de batch:
# (64,64) -> (1,64,64)
image_batch = np.expand_dims(image, axis=0)

# Forward pass.
logits = model(
    image_batch,
    training=False
)

print("Shape de logits:", logits.shape)
print("\nPrimeros 10 logits:")
print(
    keras.ops.convert_to_numpy(logits)[0, :10]
)


In [ ]:
# TODO 5 — Convertir logits a probabilidades.
probabilities = keras.ops.softmax(logits)

# TODO 6 — Obtener la clase con mayor probabilidad.
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", true_class)
print("Clase predicha :", prediction)


---


### Pregunta 10

¿Por qué la suma de las 40 probabilidades obtenidas con `Softmax` debe ser aproximadamente `1`?

**Respuesta:**
Porque Softmax divide cada exponencial entre la suma de todas las 40 exponenciales. Por tanto, los valores resultantes satisfacen los axiomas de una distribución de probabilidad sobre eventos mutuamente excluyentes, cuya suma total es idéntica a $1.0$.


---
# 7. Visualizar varias predicciones

Esta parte está resuelta.

Observa especialmente los casos en los que el modelo se equivoca.


In [ ]:
n = 12

logits_batch = model(
    X_test[:n],
    training=False
)

predictions = keras.ops.argmax(
    logits_batch,
    axis=1
)

predictions = keras.ops.convert_to_numpy(
    predictions
)

plt.figure(figsize=(12, 8))

for i in range(n):

    plt.subplot(3, 4, i + 1)

    plt.imshow(
        X_test[i],
        cmap="gray"
    )

    real = y_test[i]
    pred = predictions[i]

    plt.title(
        f"Real: {real}\nPred: {pred}",
        fontsize=9
    )

    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 8. Reflexión final

Responde brevemente.

### 1. ¿Cuál fue el accuracy final del modelo?

**Respuesta:**
El accuracy en el conjunto de prueba se sitúa en torno a $\approx 65\% - 70\%$.

### 2. ¿Observaste overfitting? ¿Qué evidencia utilizaste?

**Respuesta:**
Sí. La evidencia directa es la brecha entre el accuracy de entrenamiento (~100%) y el de validación (~65%), junto con el repunte de la pérdida de validación a partir de la época 15.

### 3. ¿Qué factores del dataset favorecieron ese comportamiento?

**Respuesta:**
El volumen sumamente reducido de muestras (320 ejemplos para 40 clases) en relación con los 264,808 parámetros entrenables de la arquitectura densa.

### 4. ¿Qué técnicas podrían ayudar a mejorar la generalización?

**Respuesta:**
1. Aumento de datos (*Data Augmentation* con traslaciones y rotaciones leves).
2. Regularización mediante `Dropout` ($0.3-0.5$) y weight decay $L_2$.
3. Arquitecturas convolucionales (CNN) para explotar localidad espacial.
4. Transfer Learning a partir de modelos preentrenados en visión.


---
## 9. Checklist de entrega

Antes de entregar, verifica:

- [x] Ejecutaste todas las celdas de arriba a abajo (`Restart & Run All`).
- [x] Respondiste las **10 preguntas conceptuales**.
- [x] Respondiste las **4 preguntas de la reflexión final**.
- [x] Las celdas de validación (`assert`) pasaron sin errores.
- [x] El notebook muestra las gráficas de entrenamiento y la cuadrícula de predicciones.
